In [16]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from astropy.io import fits
from astropy.table import Table

In [17]:
# plotting parameters
plt.rc('axes', labelsize=10)
plt.rc('axes', labelweight='regular')
plt.rc('axes', titleweight='regular')
plt.rc('axes', linewidth=1)
plt.rc('xtick',labelsize=10,direction='in',top=True)
plt.rc('ytick',labelsize=10,direction='in',right=True)
plt.rcParams['xtick.major.pad']='10'
plt.rcParams['ytick.major.pad']='10'

paper_params = {'text.usetex' : True,
           'font.size' : 14,
           'font.family' : 'lmodern',
           'figure.dpi' : 200,
           }
plt.rcParams.update(paper_params)
## 3.5 x 3 inch figure size for single column in paper
## 7 x 3 inch figure size for two column in paper

In [18]:
# read cross-matched data files
K_file = '4MOST_SAGA_Match_K.dat'
Th_file = '4MOST_SAGA_Match_Th.dat'
U_file = '4MOST_SAGA_Match_U.dat'
Eu_file = '4MOST_SAGA_Match_Eu.dat'

In [19]:
# read in the data files as astropy tables, 

# Example data:
#Found match for CS31082-001 in 4MOST catalog: 2451773941958712192. 
#                   U_RA: 22.37975, U_DEC: -16.01263888888889. 
#                   4MOST_RA: 22.37976737421, 4MOST_DEC: -16.01282775355. 
#                   U_teff: 4825, U_logg: 1.5, U_Met: -2.9, U_FeH: -2.9, U_KFe: 1.52. 
#
# so read every 4 lines as a single entry, and extract the relevant information to create a table of matched stars with their parameters
def read_matched_data(file):
    with open(file, 'r') as f:
        lines = f.readlines()
    
    data = []
    for i in range(0, len(lines), 4):
        #print(lines[i:i+4])
        entry = {}
        entry['SAGA_name'] = lines[i].split(' ')[3]
        entry['4MOST_name'] = lines[i].split(' ')[7].strip('.\n')
        entry['RA'] = float(lines[i+1].split(' ')[20].strip(','))
        entry['DEC'] = float(lines[i+1].split(' ')[22].strip('.\n'))
        entry['4MOST_RA'] = float(lines[i+2].split(' ')[20].strip(','))
        entry['4MOST_DEC'] = float(lines[i+2].split(' ')[22].strip('.\n'))
        entry['teff'] = float(lines[i+3].split(' ')[20].strip(','))
        entry['logg'] = float(lines[i+3].split(' ')[22].strip(','))
        entry['Met'] = float(lines[i+3].split(' ')[24].strip(','))

        # check if the FeH is given with uncertainties
        # if so, extract the value without the uncertainties
        if '+-' in lines[i+3].split(' ')[26]:
            entry['FeH'] = float(lines[i+3].split(' ')[26].split('+-')[0].strip(','))
            entry['FeH_err'] = float(lines[i+3].split(' ')[26].split('+-')[1].strip(','))
        elif '<' in lines[i+3].split(' ')[26]:
            entry['FeH'] = float(lines[i+3].split(' ')[26].split('<')[1].strip(','))
            entry['FeH_err'] = np.nan
        else:
            entry['FeH'] = float(lines[i+3].split(' ')[26].strip(','))
            entry['FeH_err'] = np.nan
        
        # and check for the [X/Fe] as well, which may also have uncertainties
        if '<' in lines[i+3].split(' ')[28] and '+-' in lines[i+3].split(' ')[28]:
            entry['XFe'] = float(lines[i+3].split(' ')[28].split('<')[1].split('+-')[0].strip('.\n'))
            entry['XFe_err'] = float(lines[i+3].split(' ')[28].split('<')[1].split('+-')[1].strip('.\n'))
            entry['XFe_upper_limit'] = True
        elif '+-' in lines[i+3].split(' ')[28]:
            entry['XFe'] = float(lines[i+3].split(' ')[28].split('+-')[0].strip('.\n'))
            entry['XFe_err'] = float(lines[i+3].split(' ')[28].split('+-')[1].strip('.\n'))
            entry['XFe_upper_limit'] = False
        elif '<' in lines[i+3].split(' ')[28]:
            entry['XFe'] = float(lines[i+3].split(' ')[28].split('<')[1].strip('.\n'))
            entry['XFe_err'] = np.nan
            entry['XFe_upper_limit'] = True
        else:
            entry['XFe'] = float(lines[i+3].split(' ')[28].strip('.\n'))
            entry['XFe_err'] = np.nan
            entry['XFe_upper_limit'] = False
        data.append(entry)
    
    return Table(rows=data)

In [20]:
K_stars = read_matched_data(K_file)
unique_K_stars = []
for star in K_stars:
    if star['SAGA_name'] not in [s['SAGA_name'] for s in unique_K_stars]:
        unique_K_stars.append(star)
unique_K_stars = Table(rows=unique_K_stars)
unique_K_stars

SAGA_name,4MOST_name,RA,DEC,4MOST_RA,4MOST_DEC,teff,logg,Met,FeH,FeH_err,XFe,XFe_err,XFe_upper_limit
str19,str19,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool
2M22045404-1148287,2613467159867601152,331.22516666666667,-11.807975,331.22517014284,-11.80800486128,4578.0,1.64,-1.26,-1.26,nan,0.21,0.2,False
2MJ12091322-1415313,3570484520225300352,182.30508333333336,-14.258722222222222,182.30511655999,-14.25877471063,4370.0,0.56,-2.11,-2.11,0.01,0.48,0.1,False
2MJ13254554-1747547,3604042504162287232,201.43975,-17.798527777777778,201.43960971893,-17.79864628074,4588.0,0.83,-2.32,-2.32,0.01,0.47,0.1,False
2MJ13431929-2358139,6191935642043619456,205.83041666666665,-23.970527777777775,205.83027663787,-23.97068220138,4777.0,1.13,-2.47,-2.47,0.02,0.08,0.1,False
BD-02_5957,2643630302870307328,351.71900000000005,-1.99,351.71911675052,-1.99027695815,4217.0,0.06,-3.22,-3.22,nan,0.35,0.1,False
BD-15_5781,6887351938881045504,311.39374999999995,-14.520833333333334,311.39394871828,-14.52097008239,4550.0,0.7,-2.87,-2.92,0.15,0.17,0.18,False
BD-20_6008,6857966322398347008,310.7029166666667,-20.010833333333334,310.70313906588,-20.0109723454,4540.0,0.65,-3.05,-3.0,0.15,0.32,0.18,False
CS22169-035,3189438526418585728,63.05783333333334,-12.084750000000001,63.05787558469,-12.08476501409,4700.0,1.2,-3.04,-3.04,0.19,0.51,nan,False
CS22172-029,5163769468367757568,52.68895833333333,-10.619416666666668,52.68905350217,-10.61947247917,4960.0,1.9,-2.7,-2.68,0.14,0.32,0.18,False


In [21]:
U_stars = read_matched_data(U_file)
# filter U_stars by unique name
unique_U_stars = []
for star in U_stars:
    if star['SAGA_name'] not in [s['SAGA_name'] for s in unique_U_stars]:
        unique_U_stars.append(star)
unique_U_stars = Table(rows=unique_U_stars)
unique_U_stars

SAGA_name,4MOST_name,RA,DEC,4MOST_RA,4MOST_DEC,teff,logg,Met,FeH,FeH_err,XFe,XFe_err,XFe_upper_limit
str23,str19,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool
CS22892-052,6826025986350385280,334.2569166666667,-16.657527777777776,334.25695654634,-16.65754332147,4790.0,1.6,-2.92,-2.91,0.14,1.53,nan,True
CS31082-001,2451773941958712192,22.37975,-16.01263888888889,22.37976737421,-16.01282775355,4825.0,1.5,-2.9,-2.9,nan,1.52,nan,False
HD186478,6871465473332110464,296.30875,-17.490833333333335,296.3088176197,-17.49123120322,4720.0,1.6,-2.5,-2.49,0.12,1.03,nan,True
HD6268,5033960575837265920,15.825708333333333,-27.880555555555556,15.82556743734,-27.88069223216,4600.0,1.0,-2.63,-2.62,0.11,0.53,nan,True
HE0338-3945,4856059422664301440,54.97916666666667,-39.59525,54.97927593839,-39.59527489618,6160.0,4.13,-2.42,-2.43,0.05,2.86,0.8,True
RAVEJ203843.2-002333,4226556850750887168,309.68,-0.3925,309.67995041579,-0.39250625296,4630.0,1.2,-2.91,-2.93,0.05,1.31,0.2,False
SMSSJ200322.54-114203.3,4190620966764303488,300.843875,-11.700772222222222,300.84393192485,-11.70097993036,5175.0,2.44,-3.5,-3.57,0.11,2.2,0.16,False


In [22]:
Th_stars = read_matched_data(Th_file)
unique_Th_stars = []
for star in Th_stars:
    if star['SAGA_name'] not in [s['SAGA_name'] for s in unique_Th_stars]:
        unique_Th_stars.append(star)
unique_Th_stars = Table(rows=unique_Th_stars)
unique_Th_stars

SAGA_name,4MOST_name,RA,DEC,4MOST_RA,4MOST_DEC,teff,logg,Met,FeH,FeH_err,XFe,XFe_err,XFe_upper_limit
str23,str19,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool
BD-15_5781,6887351938881045504,311.39374999999995,-14.520833333333334,311.39394871828,-14.52097008239,4550.0,0.7,-2.87,-2.92,0.15,0.13,0.36,False
BD-20_6008,6857966322398347008,310.7029166666667,-20.010833333333334,310.70313906588,-20.0109723454,4540.0,0.65,-3.05,-3.0,0.15,0.52,nan,True
CS22166-016,2372010867355028608,14.599333333333334,-14.78525,14.59948230771,-14.78531531244,4900.0,1.75,-3.09,-3.22,0.15,1.21,nan,True
CS22169-035,3189438526418585728,63.05783333333334,-12.084750000000001,63.05787558469,-12.08476501409,4480.0,0.5,-3.12,-3.31,0.15,0.98,nan,True
CS22171-037,2462280222239224704,32.00241666666667,-9.063416666666667,32.002485705,-9.06357295598,6110.0,3.65,-3.35,-3.63,0.21,3.29,nan,True
CS22172-029,5163769468367757568,52.68895833333333,-10.619416666666668,52.68905350217,-10.61947247917,4960.0,1.9,-2.7,-2.68,0.14,1.12,nan,True
CS22175-007,2486033590409430144,34.36083333333333,-9.0125,34.36097202351,-9.0126211227,5108.0,2.46,-2.81,-2.64,nan,1.51,nan,True
CS22177-009,4890881265153979904,61.91933333333333,-25.045527777777778,61.91947993123,-25.04582130279,5940.0,3.55,-3.37,-3.42,0.21,2.97,nan,True
CS22177-010,4890598793745223168,62.502,-25.74413888888889,62.50215529348,-25.7442310769,6050.0,3.7,-2.88,-3.0,0.22,2.56,nan,True


In [23]:
Eu_stars = read_matched_data(Eu_file)
unique_Eu_stars = []
for star in Eu_stars:
    if star['SAGA_name'] not in [s['SAGA_name'] for s in unique_Eu_stars]:
        unique_Eu_stars.append(star)
unique_Eu_stars = Table(rows=unique_Eu_stars)
unique_Eu_stars

SAGA_name,4MOST_name,RA,DEC,4MOST_RA,4MOST_DEC,teff,logg,Met,FeH,FeH_err,XFe,XFe_err,XFe_upper_limit
str25,str19,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool
2M22045404-1148287,2613467159867601152,331.22516666666667,-11.807975,331.22517014284,-11.80800486128,4578.0,1.64,-1.26,-1.26,nan,0.42,0.13,False
2MASSJ02031860-7930291,4632302346781151488,30.8275,-79.50808333333333,30.82750463903,-79.50810097333,4720.0,1.1,-3.03,-3.03,nan,0.36,nan,False
2MASSJ10242264-2904203,5461712380484527104,156.09433333333334,-29.072305555555555,156.09435499358,-29.07233316371,4720.0,1.3,-3.13,-3.13,nan,1.21,nan,True
2MASSJ10275895-3006391,5461371257003291008,156.995625,-30.110861111111113,156.99563190816,-30.11088035849,4648.0,1.3,-2.96,-2.96,nan,0.64,nan,True
2MJ12091322-1415313,3570484520225300352,182.30508333333336,-14.258722222222222,182.30511655999,-14.25877471063,4370.0,0.56,-2.11,-2.11,0.01,0.81,0.06,False
2MJ13254554-1747547,3604042504162287232,201.43975,-17.798527777777778,201.43960971893,-17.79864628074,4588.0,0.83,-2.32,-2.32,0.01,0.4,0.06,False
2MJ13431929-2358139,6191935642043619456,205.83041666666665,-23.970527777777775,205.83027663787,-23.97068220138,4777.0,1.13,-2.47,-2.47,0.02,0.15,nan,True
BD-02_5957,2643630302870307328,351.71900000000005,-1.99,351.71911675052,-1.99027695815,4217.0,0.06,-3.22,-3.22,nan,0.91,0.06,False
BD-11_145,2377539761575614080,12.10125,-10.691944444444445,12.10129729965,-10.69202745855,4650.0,0.7,-2.3,-2.33,0.15,0.12,0.22,False


In [24]:
# find U, Th, Eu stars with [X/Fe] > 1.0
U_enhanced = unique_U_stars[unique_U_stars['XFe'] >= 1.0]
Th_enhanced = unique_Th_stars[unique_Th_stars['XFe'] >= 1.0]
Eu_enhanced = unique_Eu_stars[unique_Eu_stars['XFe'] >= 1.0]

print(f"Number of U-enhanced stars with [X/Fe] >= 1.0: {len(U_enhanced)}")
print(f"Number of Th-enhanced stars with [X/Fe] >= 1.0: {len(Th_enhanced)}")
print(f"Number of Eu-enhanced stars with [X/Fe] >= 1.0: {len(Eu_enhanced)}")
# greater than 2.0
U_enhanced_2 = unique_U_stars[unique_U_stars['XFe'] >= 2.0]
Th_enhanced_2 = unique_Th_stars[unique_Th_stars['XFe'] >= 2.0]
Eu_enhanced_2 = unique_Eu_stars[unique_Eu_stars['XFe'] >= 2.0]

print(f"Number of U-enhanced stars with [X/Fe] >= 2.0: {len(U_enhanced_2)}")
print(f"Number of Th-enhanced stars with [X/Fe] >= 2.0: {len(Th_enhanced_2)}")
print(f"Number of Eu-enhanced stars with [X/Fe] >= 2.0: {len(Eu_enhanced_2)}")

# greater than 3.0
U_enhanced_3 = unique_U_stars[unique_U_stars['XFe'] >= 3.0]
Th_enhanced_3 = unique_Th_stars[unique_Th_stars['XFe'] >= 3.0]
Eu_enhanced_3 = unique_Eu_stars[unique_Eu_stars['XFe'] >= 3.0]

print(f"Number of U-enhanced stars with [X/Fe] >= 3.0: {len(U_enhanced_3)}")
print(f"Number of Th-enhanced stars with [X/Fe] >= 3.0: {len(Th_enhanced_3)}")
print(f"Number of Eu-enhanced stars with [X/Fe] >= 3.0: {len(Eu_enhanced_3)}")

Number of U-enhanced stars with [X/Fe] >= 1.0: 6
Number of Th-enhanced stars with [X/Fe] >= 1.0: 123
Number of Eu-enhanced stars with [X/Fe] >= 1.0: 115
Number of U-enhanced stars with [X/Fe] >= 2.0: 2
Number of Th-enhanced stars with [X/Fe] >= 2.0: 56
Number of Eu-enhanced stars with [X/Fe] >= 2.0: 11
Number of U-enhanced stars with [X/Fe] >= 3.0: 0
Number of Th-enhanced stars with [X/Fe] >= 3.0: 4
Number of Eu-enhanced stars with [X/Fe] >= 3.0: 2


In [25]:
# find most enhanced star in each category, and check the upper limit flags to see if they are robust detections or upper limits
most_enhanced_Th = np.where(Th_enhanced['XFe'] == np.max(Th_enhanced['XFe']))[0][0]
most_enhanced_Th_star = Th_enhanced[most_enhanced_Th]
print(f"Most enhanced Th star: {most_enhanced_Th_star['SAGA_name']} with [Th/Fe] = {most_enhanced_Th_star['XFe']} and upper limit flag = {most_enhanced_Th_star['XFe_upper_limit']}")
print(f"with metallicity [Fe/H] = {most_enhanced_Th_star['FeH']}")

# find most enhanced star with upper limit flag = False
most_enhanced_Th_no_upper_limit = Th_enhanced[Th_enhanced['XFe_upper_limit'] == False]
most_enhanced_Th_no_upper_limit = most_enhanced_Th_no_upper_limit[np.where(most_enhanced_Th_no_upper_limit['XFe'] == np.max(most_enhanced_Th_no_upper_limit['XFe']))[0][0]]
print(f"Most enhanced Th star with no upper limit flag: {most_enhanced_Th_no_upper_limit['SAGA_name']} with [Th/Fe] = {most_enhanced_Th_no_upper_limit['XFe']} and upper limit flag = {most_enhanced_Th_no_upper_limit['XFe_upper_limit']}")
print(f"with metallicity [Fe/H] = {most_enhanced_Th_no_upper_limit['FeH']}")

Most enhanced Th star: CS22171-037 with [Th/Fe] = 3.29 and upper limit flag = True
with metallicity [Fe/H] = -3.63
Most enhanced Th star with no upper limit flag: SMSSJ200322.54-114203.3 with [Th/Fe] = 2.2 and upper limit flag = False
with metallicity [Fe/H] = -3.57


In [26]:
# find most enhanced star in each category, and check the upper limit flags to see if they are robust detections or upper limits
most_enhanced_U = np.where(U_enhanced['XFe'] == np.max(U_enhanced['XFe']))[0][0]
most_enhanced_U_star = Th_enhanced[most_enhanced_U]
print(f"Most enhanced U star: {most_enhanced_U_star['SAGA_name']} with [U/Fe] = {most_enhanced_U_star['XFe']} and upper limit flag = {most_enhanced_U_star['XFe_upper_limit']}")
print(f"with metallicity [Fe/H] = {most_enhanced_U_star['FeH']}")
# find most enhanced star with upper limit flag = False
most_enhanced_U_no_upper_limit = U_enhanced[U_enhanced['XFe_upper_limit'] == False]
most_enhanced_U_no_upper_limit = most_enhanced_U_no_upper_limit[np.where(most_enhanced_U_no_upper_limit['XFe'] == np.max(most_enhanced_U_no_upper_limit['XFe']))[0][0]]
print(f"Most enhanced U star with no upper limit flag: {most_enhanced_U_no_upper_limit['SAGA_name']} with [U/Fe] = {most_enhanced_U_no_upper_limit['XFe']} and upper limit flag = {most_enhanced_U_no_upper_limit['XFe_upper_limit']}")
print(f"with metallicity [Fe/H] = {most_enhanced_U_no_upper_limit['FeH']}")

Most enhanced U star: CS22175-007 with [U/Fe] = 1.51 and upper limit flag = True
with metallicity [Fe/H] = -2.64
Most enhanced U star with no upper limit flag: SMSSJ200322.54-114203.3 with [U/Fe] = 2.2 and upper limit flag = False
with metallicity [Fe/H] = -3.57


In [27]:
# find most enhanced star in each category, and check the upper limit flags to see if they are robust detections or upper limits
most_enhanced_Eu = np.where(Eu_enhanced['XFe'] == np.max(Eu_enhanced['XFe']))[0][0]
most_enhanced_Eu_star = Eu_enhanced[most_enhanced_Eu]
print(f"Most enhanced Eu star: {most_enhanced_Eu_star['SAGA_name']} with [Eu/Fe] = {most_enhanced_Eu_star['XFe']} and upper limit flag = {most_enhanced_Eu_star['XFe_upper_limit']}")
print(f"with metallicity [Fe/H] = {most_enhanced_Eu_star['FeH']}")
# find most enhanced star with upper limit flag = False
most_enhanced_Eu_no_upper_limit = Eu_enhanced[Eu_enhanced['XFe_upper_limit'] == False]
most_enhanced_Eu_no_upper_limit = most_enhanced_Eu_no_upper_limit[np.where(most_enhanced_Eu_no_upper_limit['XFe'] == np.max(most_enhanced_Eu_no_upper_limit['XFe']))[0][0]]
print(f"Most enhanced Eu star with no upper limit flag: {most_enhanced_Eu_no_upper_limit['SAGA_name']} with [Eu/Fe] = {most_enhanced_Eu_no_upper_limit['XFe']} and upper limit flag = {most_enhanced_Eu_no_upper_limit['XFe_upper_limit']}")
print(f"with metallicity [Fe/H] = {most_enhanced_Eu_no_upper_limit['FeH']}")

Most enhanced Eu star: HE1327-2326 with [Eu/Fe] = 4.43 and upper limit flag = True
with metallicity [Fe/H] = -5.71
Most enhanced Eu star with no upper limit flag: CD-28_1082 with [Eu/Fe] = 2.09 and upper limit flag = False
with metallicity [Fe/H] = -2.44


In [28]:
# how many stars are enhanced in Th above +1.0 dex that are not upper limits?
Th_enhanced_no_upper_limit = Th_enhanced[Th_enhanced['XFe_upper_limit'] == False]
Th_enhanced_no_upper_limit_1 = Th_enhanced_no_upper_limit[Th_enhanced_no_upper_limit['XFe'] > 1.0]
print(f"Number of Th-enhanced stars with [Th/Fe] > 1.0 and no upper limit flag: {len(Th_enhanced_no_upper_limit_1)}")
# how many stars are enhanced in Th above +2.0 dex that are not upper limits?
Th_enhanced_no_upper_limit_2 = Th_enhanced_no_upper_limit[Th_enhanced_no_upper_limit['XFe'] > 2.0]
print(f"Number of Th-enhanced stars with [Th/Fe] > 2.0 and no upper limit flag: {len(Th_enhanced_no_upper_limit_2)}")
# how many stars are enhanced in Th above +3.0 dex that are not upper limits?
Th_enhanced_no_upper_limit_3 = Th_enhanced_no_upper_limit[Th_enhanced_no_upper_limit['XFe'] > 3.0]
print(f"Number of Th-enhanced stars with [Th/Fe] > 3.0 and no upper limit flag: {len(Th_enhanced_no_upper_limit_3)}")

# how many stars are enhanced in Th above +1.0 dex that are upper limits?
Th_enhanced_with_upper_limit = Th_enhanced[Th_enhanced['XFe_upper_limit'] == True]
Th_enhanced_with_upper_limit_1 = Th_enhanced_with_upper_limit[Th_enhanced_with_upper_limit['XFe'] > 1.0]
print(f"Number of Th-enhanced stars with [Th/Fe] > 1.0 and upper limit flag: {len(Th_enhanced_with_upper_limit_1)}")
# how many stars are enhanced in Th above +2.0 dex that are upper limits?
Th_enhanced_with_upper_limit_2 = Th_enhanced_with_upper_limit[Th_enhanced_with_upper_limit['XFe'] > 2.0]
print(f"Number of Th-enhanced stars with [Th/Fe] > 2.0 and upper limit flag: {len(Th_enhanced_with_upper_limit_2)}")
# how many stars are enhanced in Th above +3.0 dex that are upper limits?
Th_enhanced_with_upper_limit_3 = Th_enhanced_with_upper_limit[Th_enhanced_with_upper_limit['XFe'] > 3.0]
print(f"Number of Th-enhanced stars with [Th/Fe] > 3.0 and upper limit flag: {len(Th_enhanced_with_upper_limit_3)}")

Number of Th-enhanced stars with [Th/Fe] > 1.0 and no upper limit flag: 13
Number of Th-enhanced stars with [Th/Fe] > 2.0 and no upper limit flag: 1
Number of Th-enhanced stars with [Th/Fe] > 3.0 and no upper limit flag: 0
Number of Th-enhanced stars with [Th/Fe] > 1.0 and upper limit flag: 110
Number of Th-enhanced stars with [Th/Fe] > 2.0 and upper limit flag: 55
Number of Th-enhanced stars with [Th/Fe] > 3.0 and upper limit flag: 4


In [29]:
# how many stars are enhanced in U above +1.0 dex that are not upper limits?
U_enhanced_no_upper_limit = U_enhanced[U_enhanced['XFe_upper_limit'] == False]
U_enhanced_no_upper_limit_1 = U_enhanced_no_upper_limit[U_enhanced_no_upper_limit['XFe'] > 1.0]
print(f"Number of U-enhanced stars with [U/Fe] > 1.0 and no upper limit flag: {len(U_enhanced_no_upper_limit_1)}")
# how many stars are enhanced in U above +2.0 dex that are not upper limits?
U_enhanced_no_upper_limit_2 = U_enhanced_no_upper_limit[U_enhanced_no_upper_limit['XFe'] > 2.0]
print(f"Number of U-enhanced stars with [U/Fe] > 2.0 and no upper limit flag: {len(U_enhanced_no_upper_limit_2)}")
# how many stars are enhanced in U above +3.0 dex that are not upper limits?
U_enhanced_no_upper_limit_3 = U_enhanced_no_upper_limit[U_enhanced_no_upper_limit['XFe'] > 3.0]
print(f"Number of U-enhanced stars with [U/Fe] > 3.0 and no upper limit flag: {len(U_enhanced_no_upper_limit_3)}")

# how many stars are enhanced in U above +1.0 dex that are upper limits?
U_enhanced_with_upper_limit = U_enhanced[U_enhanced['XFe_upper_limit'] == True]
U_enhanced_with_upper_limit_1 = U_enhanced_with_upper_limit[U_enhanced_with_upper_limit['XFe'] > 1.0]
print(f"Number of U-enhanced stars with [U/Fe] > 1.0 and upper limit flag: {len(U_enhanced_with_upper_limit_1)}")
# how many stars are enhanced in U above +2.0 dex that are upper limits?
U_enhanced_with_upper_limit_2 = U_enhanced_with_upper_limit[U_enhanced_with_upper_limit['XFe'] > 2.0]
print(f"Number of U-enhanced stars with [U/Fe] > 2.0 and upper limit flag: {len(U_enhanced_with_upper_limit_2)}")
# how many stars are enhanced in U above +3.0 dex that are upper limits?
U_enhanced_with_upper_limit_3 = U_enhanced_with_upper_limit[U_enhanced_with_upper_limit['XFe'] > 3.0]
print(f"Number of U-enhanced stars with [U/Fe] > 3.0 and upper limit flag: {len(U_enhanced_with_upper_limit_3)}")

Number of U-enhanced stars with [U/Fe] > 1.0 and no upper limit flag: 3
Number of U-enhanced stars with [U/Fe] > 2.0 and no upper limit flag: 1
Number of U-enhanced stars with [U/Fe] > 3.0 and no upper limit flag: 0
Number of U-enhanced stars with [U/Fe] > 1.0 and upper limit flag: 3
Number of U-enhanced stars with [U/Fe] > 2.0 and upper limit flag: 1
Number of U-enhanced stars with [U/Fe] > 3.0 and upper limit flag: 0


In [30]:
# how many stars are enhanced in Eu above +1.0 dex that are not upper limits?
Eu_enhanced_no_upper_limit = Eu_enhanced[Eu_enhanced['XFe_upper_limit'] == False]
Eu_enhanced_no_upper_limit_1 = Eu_enhanced_no_upper_limit[Eu_enhanced_no_upper_limit['XFe'] > 1.0]
print(f"Number of Eu-enhanced stars with [Eu/Fe] > 1.0 and no upper limit flag: {len(Eu_enhanced_no_upper_limit_1)}")
# how many stars are enhanced in Eu above +2.0 dex that are not upper limits?
Eu_enhanced_no_upper_limit_2 = Eu_enhanced_no_upper_limit[Eu_enhanced_no_upper_limit['XFe'] > 2.0]
print(f"Number of Eu-enhanced stars with [Eu/Fe] > 2.0 and no upper limit flag: {len(Eu_enhanced_no_upper_limit_2)}")
# how many stars are enhanced in Eu above +3.0 dex that are not upper limits?
Eu_enhanced_no_upper_limit_3 = Eu_enhanced_no_upper_limit[Eu_enhanced_no_upper_limit['XFe'] > 3.0]
print(f"Number of Eu-enhanced stars with [Eu/Fe] > 3.0 and no upper limit flag: {len(Eu_enhanced_no_upper_limit_3)}")

# how many stars are enhanced in Eu above +1.0 dex that are upper limits?
Eu_enhanced_with_upper_limit = Eu_enhanced[Eu_enhanced['XFe_upper_limit'] == True]
Eu_enhanced_with_upper_limit_1 = Eu_enhanced_with_upper_limit[Eu_enhanced_with_upper_limit['XFe'] > 1.0]
print(f"Number of Eu-enhanced stars with [Eu/Fe] > 1.0 and upper limit flag: {len(Eu_enhanced_with_upper_limit_1)}")
# how many stars are enhanced in Eu above +2.0 dex that are upper limits?
Eu_enhanced_with_upper_limit_2 = Eu_enhanced_with_upper_limit[Eu_enhanced_with_upper_limit['XFe'] > 2.0]
print(f"Number of Eu-enhanced stars with [Eu/Fe] > 2.0 and upper limit flag: {len(Eu_enhanced_with_upper_limit_2)}")
# how many stars are enhanced in Eu above +3.0 dex that are upper limits?
Eu_enhanced_with_upper_limit_3 = Eu_enhanced_with_upper_limit[Eu_enhanced_with_upper_limit['XFe'] > 3.0]
print(f"Number of Eu-enhanced stars with [Eu/Fe] > 3.0 and upper limit flag: {len(Eu_enhanced_with_upper_limit_3)}")

Number of Eu-enhanced stars with [Eu/Fe] > 1.0 and no upper limit flag: 42
Number of Eu-enhanced stars with [Eu/Fe] > 2.0 and no upper limit flag: 1
Number of Eu-enhanced stars with [Eu/Fe] > 3.0 and no upper limit flag: 0
Number of Eu-enhanced stars with [Eu/Fe] > 1.0 and upper limit flag: 71
Number of Eu-enhanced stars with [Eu/Fe] > 2.0 and upper limit flag: 9
Number of Eu-enhanced stars with [Eu/Fe] > 3.0 and upper limit flag: 2


In [ ]:
# find overlap between the Eu, Th, and U star samples. search by name and coordinates for a master list of stars enriched in each element:
